<h2>Lab Report #1: Statistical Properties of Stock Returns</h2>
RSM338<br>
Prof. K. Mott<br>
Submitted by Dong-Hyun Michael Kim<br>
2026-02-04

<h2>Problem 1: Monthly Size Portfolio Returns</h2>

<h3>Data Preparation</h3>
Convert the given csv files into pandas dataframes in Python, and create a combined dataframe of the return series of the entire market as well as portfolios of deciles of the market by size (all expressed in log returns)

In [ ]:
import pandas as pd
import numpy as np

# File paths
mo_fff_return_file = 'F-F_Research_Data_Factors.txt'
mo_portfolio_return_file = 'Portfolios_Formed_on_ME.txt'

# Read the file, skipping header lines until the data starts
factors_df = pd.read_csv(
    mo_fff_return_file,
    sep='\s+',                # whitespace separator
    skiprows=4,               # usually 3, but check your file for the correct number
    nrows=1193,                # number of data rows; adjust as needed
    index_col=0               # first column as index (e.g., date)
)

port_return_df = pd.read_csv(
    mo_portfolio_return_file,
    sep='\s{2,}',  # two or more spaces as separator
    skiprows=12,
    nrows=1193,
    index_col=0
)

# Ensure columns are numeric before subtraction
factors_df['Mkt-RF'] = pd.to_numeric(factors_df['Mkt-RF'], errors='coerce')
factors_df['RF'] = pd.to_numeric(factors_df['RF'], errors='coerce')
mkt_r = factors_df['Mkt-RF'] + factors_df['RF']

# Select the last 10 columns from port_return_df
last_10 = port_return_df.iloc[:, -10:]

# Convert both indices to string type for consistency
mkt_r.index = mkt_r.index.astype(str)
last_10.index = last_10.index.astype(str)

# Align last_10 to mkt_r's index
last_10_aligned = last_10.reindex(mkt_r.index)

# Concatenate and drop rows with any NaN values
combined_df = pd.concat([mkt_r, last_10_aligned], axis=1)

# Convert to log returns
log_returns_df = np.log1p(combined_df / 100)

<h3>Task a: Summary Statistics</h3>
Compute the mean, standard deviation, and quartiles of the 11 portfolios (market and deciles)

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

def task_a(df, pltnum, dot_colour='b', line_colour='r', label=""):
    # Compute summary statistics for log_returns_df (mean, std, and quartiles only)
    summary_stats = df.describe().loc[['mean', 'std', '25%', '50%', '75%']]

    print(summary_stats)

    # Extract the last 10 columns' mean values
    mean_returns = summary_stats.iloc[0, -10:]

    deciles = range(1, 11)

    # Add a constant term for the intercept
    X = sm.add_constant(range(1, 11))  # Adds a column of ones to the design matrix
    y = mean_returns

    # OLS regression
    model = sm.OLS(y, X).fit()

    # Create a plot
    plt.figure(num=pltnum, figsize=(8, 5))
    plt.plot(deciles, mean_returns, marker='o', linestyle='', color=dot_colour, label=label)
    plt.plot(deciles, model.fittedvalues, color=line_colour)
    plt.title("Mean Log Returns by Decile")
    plt.xlabel('Decile Number')
    plt.ylabel('Mean Log Return')
    plt.xticks(range(1, 11))  # Ensure x-axis shows decile numbers 1 to 10
    plt.grid(True)

print("Summary Statistics and OLS Regression for All Months:")
task_a(log_returns_df, 1)
plt.show()

The line of best fit suggests that there is a negative relationship between firm size and average returns, but it is not strong as there are some significant outliers.

<h3>Task b: normality test</h3>

In [ ]:
from scipy.stats import skew, kurtosis, jarque_bera

def task_b(df):
    # Compute skewness for each column
    skewness = df.apply(skew)

    # Compute excess kurtosis for each column (subtract 3 to get excess kurtosis)
    excess_kurtosis = df.apply(kurtosis)  # By default, kurtosis() gives excess kurtosis

    # Combine results into a DataFrame
    stats_df = pd.DataFrame({
        'Skewness': skewness,
        'Excess Kurtosis': excess_kurtosis
    })

    print(stats_df)

    # Perform the Jarque-Bera test for each column
    normality_results = df.apply(lambda x: jarque_bera(x), axis=0)

    # Extract the test statistics and p-values
    jb_stats = normality_results.apply(lambda x: x[0])  # Test statistic
    p_values = normality_results.apply(lambda x: x[1])  # p-value

    # Combine results into a DataFrame
    normality_df = pd.DataFrame({
        'Jarque-Bera Statistic': jb_stats,
        'p-value': p_values
    })

    print()
    print(normality_df)

print("Skewness, Excess Kurtosis, and Jarque-Bera Test for All Months:")
task_b(log_returns_df)

Here we use the Jarque-Bera test for normality, as it is computed using sample skewness and excess kurtosis. Notably, each portfolio has a large number as its Jarque-Bera statistic, with corresponding p-value near zero. Thus, we reject normality for all 11 portfolios.

<h3>Task c: January effect</h3>
Split up the dataframe into two sets: one for return data for each January of each year, and one for return data for every other month.

In [ ]:
january_log_returns_df = log_returns_df.loc[log_returns_df.index.str.endswith('01')]

non_jan_log_returns_df = log_returns_df.loc[~log_returns_df.index.str.endswith('01')]

print("Summary Statistics and OLS Regression for January Data:")
task_a(january_log_returns_df, pltnum=2, dot_colour='b', line_colour='b', label="January")
print("\nSkewness, Excess Kurtosis, and Jarque-Bera Test for January Data:")
task_b(january_log_returns_df)

print("\n\nSummary Statistics and OLS Regression for Non-January Data:")
task_a(non_jan_log_returns_df, pltnum=2, dot_colour='r', line_colour='r', label="Non-January")
print("\nSkewness, Excess Kurtosis, and Jarque-Bera Test for Non-January Data:")
task_b(non_jan_log_returns_df)
plt.legend(loc='upper right')
plt.show()

Here we see a staunch difference between January returns and non-January returns, with the difference being roughly inversely proportional to size.

<h3>Task d </h3>
For any given decile, value-weighted means you have a portfolio of all the firms in the decile except the amount invested in each firm is proportional to its size. This means value-weighted returns are more sensitive to changes in the returns of larger firms within the decile. In contrast, equal weight means a portfolio where an equal investment is made in all the firms within the decile.

<h2>Problem 2: Constructing Weekly Returns from Daily Data</h2>

<h3>Data Preparation</h3>
Just like problem 1, convert the given csv into a pandas dataframe and express market returns as log returns.

In [ ]:
# File path
da_fff_return_file = 'F-F_Research_Data_Factors_daily.txt'

# Read the file, skipping header lines until the data starts
factors_df = pd.read_csv(
    da_fff_return_file,
    sep='\s+',                # whitespace separator
    skiprows=4,               # usually 3, but check your file for the correct number
    nrows=26129,                # number of data rows; adjust as needed
    index_col=0               # first column as index (e.g., date)
)

# Ensure columns are numeric before subtraction
factors_df['Mkt-RF'] = pd.to_numeric(factors_df['Mkt-RF'], errors='coerce')
factors_df['RF'] = pd.to_numeric(factors_df['RF'], errors='coerce')
mkt_r = factors_df['Mkt-RF'] + factors_df['RF']
mkt_r.index = pd.to_datetime(mkt_r.index, format='%Y%m%d', errors='coerce')

# Convert to log returns
log_returns_df = np.log1p(mkt_r / 100)

<h3>Task a: Aggregate Returns to Weekly Frequency</h3>
Where each weekly return is the sum of its daily returns.

In [133]:
# Count the number of non-NaN values in each week
weekly_counts = log_returns_df.resample('W').count()

# Filter weeks with at least 5 days of data
valid_weeks = weekly_counts[weekly_counts >= 5].dropna()

# Resample log_returns_df to weekly frequency and sum the log returns for valid weeks
weekly_log_returns_df = log_returns_df.resample('W').sum().loc[valid_weeks.index]
weekly_log_returns_df = weekly_log_returns_df.to_frame(name='Mkt Returns')

print(weekly_log_returns_df)

            Mkt Returns
1926-07-11     0.004251
1926-07-18     0.010459
1926-07-25    -0.019959
1926-08-01     0.030797
1926-08-08     0.020810
...                 ...
2025-10-26     0.019561
2025-11-02     0.006559
2025-11-09    -0.017161
2025-11-16    -0.000477
2025-11-23    -0.019263

[4478 rows x 1 columns]


Fortunately pandas makes this task very simple. In this report we include all weeks with 5 or more trading days. This is to ensure consistency within the data, as weeks affected by holidays will have lower aggregate returns and pull statistics downwards.

<h3>Task b: Summary Statistics</h3>
Just like in the previous problem, compute the mean, standard deviation, quartiles, skewness, and excess kurtosis of the weekly return data. Then test its normality using the Jarque-Bera test.

In [134]:
# For weekly_log_returns_df (single column), just print summary stats and skip OLS regression
print(weekly_log_returns_df.describe().loc[['mean', 'std', '25%', '50%', '75%']])
print()
task_b(weekly_log_returns_df)

      Mkt Returns
mean     0.001526
std      0.025354
25%     -0.010315
50%      0.003567
75%      0.014821

             Skewness  Excess Kurtosis
Mkt Returns -0.697549         7.352104

             Jarque-Bera Statistic  p-value
Mkt Returns           10448.616164      0.0


In this case as well the Jarque-Bera statistic is very high and so we reject normality for weekly returns.

<h3>Task d</h3>
When comparing the two, weekly returns are less normal than monthly returns. This makes sense since given the Central Limit Theorem, normality should increase as the return horizon increases.

<h4>Sources:</h4>GPT-4.1 and 4o<br>
Used for all coding tasks, but not for analysis. Annoyingly 4.1 could not figure out what was wrong in P1 Task a when I was merging the dataframes and the indices weren't matching. 4o seems to fare better, but also fails to understand errors from time to time.